In [ ]:
import os
import tarfile

def compress_parquet_folder(input_folder, output_file):
    """
    Compress all parquet files in a folder into a tar.gz archive.
    """
    with tarfile.open(output_file, "w:gz") as tar:
        for root, dirs, files in os.walk(input_folder):
            for file in files:
                if file.endswith(".parquet"):
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, input_folder)
                    
                    print(f"Adding: {arcname}")
                    tar.add(full_path, arcname=arcname)

    print(f"\nDone! Archive created: {output_file}")



compress_parquet_folder("trades", "trades.tar.gz")
#compress_parquet_folder("open_interest", "open_interest.tar.gz")
compress_parquet_folder("liquidations", "liquidations.tar.gz")
#compress_parquet_folder("orderbook", "trades.tar.gz")

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])
subprocess.run([sys.executable, "-m", "pip", "install", "polars"])

In [ ]:
"""
Download the Parquet files Single Core.
"""

import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc  # <--- Important for memory management

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-08-01"
END_DATE = "2025-08-30" 
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES

DROP_HEADERS = ['some_unused_column', 'internal_id', 'order_count', 'transaction_time', 'event_time', 'timestamp', 'first_update_id', 'final_update_id', 'prev_final_update_id', 'last_update_id'] 

client = chd.CryptoHFTDataClient(api_key=API_KEY)

def optimize_floats(df):
    """Downcast floats to save 50% memory."""
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df

def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(delta.days + 1)]

def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)
    categories = [
        #("trades", client.get_trades),
        #("open_interest", client.get_open_interest),
        #("liquidations", client.get_liquidations),
        ("orderbook", client.get_orderbook)
    ]

    for cat_name, fetch_func in categories:
        print(f"\n>>> Starting Category: {cat_name.upper()}")
        os.makedirs(cat_name, exist_ok=True)

        for date_str in dates:
            file_path = f"{cat_name}/{date_str}_{SYMBOL}.parquet"

            if os.path.exists(file_path):
                print(f"  [SKIP] {date_str} already exists.")
                continue

            try:
                print(f"  [FETCH] {date_str}...", end="\r")
                df = fetch_func(
                    symbol=SYMBOL,
                    exchange=EXCHANGE,
                    start_date=date_str,
                    end_date=date_str
                )

                if df is not None and not df.empty:
                    # 1. Drop unused columns immediately
                    df.drop(columns=[c for c in DROP_HEADERS if c in df.columns], errors='ignore', inplace=True)

                    # 2. Convert time
                    if 'received_time' in df.columns:
                        df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

                    # 3. Optimize Memory usage (Shrink floats)
                    df = optimize_floats(df)

                    # 4. Save to Disk
                    df.to_parquet(file_path, engine='pyarrow', compression='zstd', compression_level=9, index=False)
                    
                    print(f"  [SAVED] {date_str} | Rows: {len(df):,}")
                    
                    # 5. AGGRESSIVE MEMORY CLEANUP
                    del df
                    gc.collect() # Manually trigger garbage collection
                else:
                    print(f"  [EMPTY] {date_str} - No data found.")

            except Exception as e:
                print(f"\n  [ERROR] {date_str}: {e}")
                # Clean up even on error to prevent leaks
                if 'df' in locals(): del df
                gc.collect()

if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

In [ ]:
"""
Download the Parquet files Multi Core.
"""

import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-08-01"
END_DATE = "2025-08-30"
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES
MAX_WORKERS = 6  # <- tune this (start low!)

DROP_HEADERS = [
    'some_unused_column', 'internal_id', 'order_count',
    'transaction_time', 'event_time', 'timestamp',
    'first_update_id', 'final_update_id',
    'prev_final_update_id', 'last_update_id'
]


def optimize_floats(df):
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df


def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(delta.days + 1)]


def process_single_date(date_str, category):
    """Runs inside each process."""
    client = chd.CryptoHFTDataClient(api_key=API_KEY)

    cat_name, fetch_method_name = category
    fetch_func = getattr(client, fetch_method_name)

    os.makedirs(cat_name, exist_ok=True)
    file_path = f"{cat_name}/{date_str}_{SYMBOL}.parquet"

    if os.path.exists(file_path):
        return f"[SKIP] {date_str}"

    try:
        df = fetch_func(
            symbol=SYMBOL,
            exchange=EXCHANGE,
            start_date=date_str,
            end_date=date_str
        )

        if df is None or df.empty:
            return f"[EMPTY] {date_str}"

        # Drop unused columns
        df.drop(columns=[c for c in DROP_HEADERS if c in df.columns],
                errors='ignore', inplace=True)

        # Convert time
        if 'received_time' in df.columns:
            df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

        # Optimize
        df = optimize_floats(df)

        # Save
        df.to_parquet(
            file_path,
            engine='pyarrow',
            compression='zstd',
            compression_level=9,
            index=False
        )

        rows = len(df)

        del df
        gc.collect()

        return f"[SAVED] {date_str} | Rows: {rows:,}"

    except Exception as e:
        if 'df' in locals():
            del df
        gc.collect()
        return f"[ERROR] {date_str}: {e}"


def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)

    categories = [
        #("orderbook", "get_orderbook"),
        ("trades", "get_trades"),
        ("open_interest", "get_open_interest"),
    ]

    for category in categories:
        cat_name, _ = category
        print(f"\n>>> Starting Category: {cat_name.upper()}")

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_date, date, category)
                for date in dates
            ]

            for future in as_completed(futures):
                print(" ", future.result())


if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

In [ ]:
import polars as pl
import pandas as pd
from sortedcontainers import SortedDict
from pathlib import Path
import pickle
import os
from datetime import datetime, timezone, timedelta

def negate(x):
    return -x

def reconstruct_and_featurize(filepath: str, output_dir: str, state_path: str, resample_interval: str = "30s"):
    # 1. Load State
    if os.path.exists(state_path):
        with open(state_path, "rb") as f:
            bids, asks = pickle.load(f)
    else:
        bids, asks = SortedDict(negate), SortedDict()

    # 2. Polars Processing (Fast)
    df = pl.read_parquet(filepath)
    df = (
        df.with_columns([
            pl.col("price").cast(pl.Float64),
            pl.col("quantity").cast(pl.Float64),
            pl.col("received_time").dt.truncate("1s").dt.replace_time_zone("UTC").alias("ts_1s")
        ])
        .sort(["ts_1s", "received_time"]) # Strict sorting
        .group_by(["ts_1s", "side", "price"], maintain_order=True)
        .agg(pl.col("quantity").last())
        .sort("ts_1s")
    )

    # 3. Optimized Loop: Use column-wise access instead of iter_rows
    # Extracting to python lists is faster for the loop than row-access
    times = df["ts_1s"].to_list()
    sides = df["side"].to_list()
    prices = df["price"].to_list()
    quants = df["quantity"].to_list()

    # Pre-allocate containers (Faster than list of dicts)
    # Using a 2D array or multiple lists
    results = []
    
    current_ts = None
    
    # helper to extract features
    def get_snapshot(ts):
        if not bids or not asks: return None
    
        try:
            # Get up to 10 items (price, qty) from each side
            # bids.items() is already sorted because it's a SortedDict
            b_items = [bids.peekitem(i) for i in range(min(10, len(bids)))]
            a_items = [asks.peekitem(i) for i in range(min(10, len(asks)))]

            # bids prices are negative, so -p[0] makes them positive
            b_prices = [p[0]  for p in b_items]
            b_quants = [p[1] for p in b_items]
            
            a_prices = [p[0] for p in a_items]
            a_quants = [p[1] for p in a_items]

            best_bid, best_ask = b_prices[0], a_prices[0]
            
            # Calculate volumes
            b_vol_5, a_vol_5 = sum(b_quants[:5]), sum(a_quants[:5])
            b_vol_10, a_vol_10 = sum(b_quants), sum(a_quants)
            
            mid = (best_bid + best_ask) / 2
            spread = best_ask - best_bid
            
            return (
                ts, mid, spread, (spread / best_bid * 10000),
                (b_vol_5 - a_vol_5) / (b_vol_5 + a_vol_5) if (b_vol_5 + a_vol_5) > 0 else 0,
                (b_vol_10 - a_vol_10) / (b_vol_10 + a_vol_10) if (b_vol_10 + a_vol_10) > 0 else 0,
                b_vol_5, a_vol_5
            )
        except (IndexError, ZeroDivisionError):
            return None


    for i in range(len(times)):
        ts, side, price, qty = times[i], sides[i], prices[i], quants[i]
        
        # Update Book
        book = bids if side == "bid" else asks
        if qty == 0:
            book.pop(price, None)
        else:
            book[price] = qty
            
        # Only take snapshot when the second changes OR at the last row
        if i + 1 < len(times):
            if times[i+1] != ts:
                snap = get_snapshot(ts)
                if snap: results.append(snap)
        else:
            snap = get_snapshot(ts)
            if snap: results.append(snap)

    # 4. Convert back to Polars for resampling (Massive Speedup)
    res_df = pl.DataFrame(results, schema=[
        "time", "mid_price", "spread", "spread_bps", "obi_5", "obi_10", "bid_vol_5", "ask_vol_5"
    ], orient="row")

    # Generate full time range and join to fill gaps (Polars Style)
    day_str = Path(filepath).stem[:10]
    start = datetime.strptime(day_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = start + timedelta(days=1) - timedelta(seconds=1)
    
    full_grid = pl.datetime_range(start, end, interval="1s", eager=True).alias("time").to_frame()
    
    final_df = (
        full_grid.join(res_df, on="time", how="left")
        .fill_null(strategy="forward")
        .drop_nulls() # drop leading nulls if no state
    )

    # 5. Resample using Polars group_by_dynamic
    # Much faster than Pandas resample
    output = (
        final_df.group_by_dynamic("time", every=resample_interval)
        .agg([
            pl.col("mid_price").first().alias("mid_price_first"),
            pl.col("mid_price").last().alias("mid_price_last"),
            pl.col("mid_price").mean().alias("mid_price_mean"),
            pl.col("spread_bps").mean().alias("spread_bps_mean"),
            pl.col("spread_bps").max().alias("spread_bps_max"),
            pl.col("obi_5").mean().alias("obi_5_mean"),
            pl.col("obi_5").std().alias("obi_5_std"),
            pl.col("obi_5").last().alias("obi_5_last"),
            pl.col("obi_10").mean().alias("obi_10_mean"),
            pl.col("obi_10").last().alias("obi_10_last"),
            pl.col("bid_vol_5").mean().alias("bid_vol_5_mean"),
            pl.col("ask_vol_5").mean().alias("ask_vol_5_mean"),
        ])
    )

    # Save state
    os.makedirs(Path(state_path).parent, exist_ok=True)
    with open(state_path, "wb") as f:
        pickle.dump((bids, asks), f)

    out_path = Path(output_dir) / (Path(filepath).stem + "_features.parquet")
    output.write_parquet(out_path)


def process_all(input_dir: str, output_dir: str, state_path: str = "ob_state.pkl"):
    files = sorted(Path(input_dir).glob("*.parquet"))  # sort → chronologische Reihenfolge
    os.makedirs(output_dir, exist_ok=True)

    for f in files:
        print(f"Verarbeite: {f.name}")
        reconstruct_and_featurize(str(f), output_dir, state_path)


if __name__ == "__main__":
    process_all(
        input_dir  = "orderbook/",
        output_dir = "features/",
        state_path = "tmp/ob_state.pkl",
    )

In [ ]:
#Optimized Version of the Order Book Reconstruction and Featurization
# Key idea: remove Python loop bottleneck, minimize SortedDict overhead,
# and push as much as possible into vectorized / compiled paths.

import polars as pl
import numpy as np
from pathlib import Path
import pickle
import os
from datetime import datetime, timezone, timedelta

# -----------------------------
# Faster book representation
# -----------------------------
# Replace SortedDict with two arrays (price, qty) + binary search
# This avoids Python-level tree overhead

class OrderBookSide:
    def __init__(self, is_bid=False):
        self.prices = []
        self.qty = []
        self.is_bid = is_bid

    def update(self, price, quantity):
        import bisect

        idx = bisect.bisect_left(self.prices, price)

        if idx < len(self.prices) and self.prices[idx] == price:
            if quantity == 0:
                self.prices.pop(idx)
                self.qty.pop(idx)
            else:
                self.qty[idx] = quantity
        else:
            if quantity != 0:
                self.prices.insert(idx, price)
                self.qty.insert(idx, quantity)

    def top_n(self, n=10):
        if self.is_bid:
            # reverse for bids
            p = self.prices[-n:][::-1]
            q = self.qty[-n:][::-1]
        else:
            p = self.prices[:n]
            q = self.qty[:n]
        return p, q

# -----------------------------
# Main function
# -----------------------------

def reconstruct_and_featurize(filepath: str, output_dir: str, state_path: str):

    # Load state
    if os.path.exists(state_path):
        with open(state_path, "rb") as f:
            bids, asks = pickle.load(f)
    else:
        bids, asks = OrderBookSide(True), OrderBookSide(False)

    # Lazy scan instead of eager read (huge win for large files)
    df = (
        pl.scan_parquet(filepath)
        .with_columns([
            pl.col("price").cast(pl.Float64),
            pl.col("quantity").cast(pl.Float64),
            pl.col("received_time").dt.truncate("1s").alias("ts_1s")
        ])
        .sort(["ts_1s", "received_time"])
        .group_by(["ts_1s", "side", "price"])
        .agg(pl.col("quantity").last())
        .collect(streaming=True)  # streaming execution
    )

    times = df["ts_1s"].to_numpy()
    sides = df["side"].to_numpy()
    prices = df["price"].to_numpy()
    quants = df["quantity"].to_numpy()

    results = []

    def snapshot(ts):
        if len(bids.prices) == 0 or len(asks.prices) == 0:
            return None

        b_p, b_q = bids.top_n(10)
        a_p, a_q = asks.top_n(10)

        best_bid = b_p[0]
        best_ask = a_p[0]

        b5 = sum(b_q[:5])
        a5 = sum(a_q[:5])
        b10 = sum(b_q)
        a10 = sum(a_q)

        mid = (best_bid + best_ask) / 2
        spread = best_ask - best_bid

        return (
            ts, mid, spread, spread / best_bid * 10000,
            (b5 - a5) / (b5 + a5) if (b5 + a5) else 0,
            (b10 - a10) / (b10 + a10) if (b10 + a10) else 0,
            b5, a5
        )

    # Tight loop (now much faster due to simpler structure)
    for i in range(len(times)):
        if sides[i] == "bid":
            bids.update(prices[i], quants[i])
        else:
            asks.update(prices[i], quants[i])

        if i == len(times) - 1 or times[i + 1] != times[i]:
            snap = snapshot(times[i])
            if snap:
                results.append(snap)

    res_df = pl.DataFrame(results, schema=[
        "time", "mid_price", "spread", "spread_bps",
        "obi_5", "obi_10", "bid_vol_5", "ask_vol_5"
    ])

    # Time grid (vectorized, no Python loop)
    day_str = Path(filepath).stem[:10]
    start = datetime.strptime(day_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = start + timedelta(days=1) - timedelta(seconds=1)

    full_grid = pl.datetime_range(start, end, interval="1s", eager=True).to_frame("time")

    final_df = (
        full_grid.join(res_df, on="time", how="left")
        .fill_null(strategy="forward")
        .drop_nulls()
    )

    output = (
        final_df.group_by_dynamic("time", every="30s")
        .agg([
            pl.col("mid_price").first(),
            pl.col("mid_price").last(),
            pl.col("mid_price").mean(),
            pl.col("spread_bps").mean(),
            pl.col("obi_5").mean(),
        ])
    )

    os.makedirs(Path(state_path).parent, exist_ok=True)
    with open(state_path, "wb") as f:
        pickle.dump((bids, asks), f)

    out_path = Path(output_dir) / (Path(filepath).stem + "_features.parquet")
    output.write_parquet(out_path)


# -----------------------------
# Parallelism where it ACTUALLY works
# -----------------------------
# Files are sequential, but inside each file Polars already uses multithreading.
# So DO NOT fight it — feed it large chunks.


def process_all(input_dir: str, output_dir: str, state_path: str = "ob_state.pkl"):
    files = sorted(Path(input_dir).glob("*.parquet"))
    os.makedirs(output_dir, exist_ok=True)

    for f in files:
        print(f"Processing: {f.name}")
        reconstruct_and_featurize(str(f), output_dir, state_path)


if __name__ == "__main__":
    process_all("orderbook/", "features/", "tmp/ob_state.pkl")
